# Silver — customers

Bronze `customers` → `silver.customers` (SCD type 2).

What the CRM extract needs fixing:

| Problem found in profiling | What we do |
| --- | --- |
| `customer_id` repeats — the CRM keeps its own versions (`valid_from`/`valid_to`) | keep the latest source version per customer |
| text `'NULL'`, `'NA NA'`, `'nan'` instead of real NULLs | normalise to NULL |
| `46506.0` postcodes and house numbers (numbers that went via float) | strip the `.0` |
| `valid_from`/`valid_to` are epoch seconds | cast to timestamps |
| `customer_name` is `LAST, FIRST` for people, plain text for companies | derive type, first and last name |
| loyalty is a code 0–3 | add the label marketing uses |

SCD2 because reports must be able to say where a customer lived *at the time*,
not only where they live now.

In [ ]:
import sys
from datetime import date
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "common_utils").is_dir():
        sys.path.insert(0, str(candidate))
        break

from pyspark.sql import functions as F

from common_utils.logger import get_logger, log_info
from common_utils.observability import ensure_ops_schema, new_run_id, track
from common_utils.scd import business_columns, deduplicate, row_hash, scd2_merge
from common_utils.settings import parse_run_date
from common_utils.transforms import add_derived, cast_columns, drop_columns, normalise_nulls, rename_columns, strip_float_suffix, trim_columns
from common_utils.writers import cluster_by, create_namespace, qualified, set_table_properties

In [ ]:
dbutils.widgets.text("catalog", "retaildataplatform")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("run_date", date.today().isoformat())
dbutils.widgets.text("detect_deletes", "false")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
run_date = parse_run_date(dbutils.widgets.get("run_date"))
detect_deletes = dbutils.widgets.get("detect_deletes").lower() == "true"
run_id = new_run_id()
logger = get_logger("silver")

create_namespace(spark, catalog, silver_schema, comment="Silver: cleaned, typed and de-duplicated entities with change history")
ensure_ops_schema(spark, catalog)

## 1. Read this run's Bronze rows
Only `run_date` — Bronze holds every day, Silver processes one.

In [ ]:
bronze = spark.table(f"{catalog}.{bronze_schema}.customers").filter(F.col("_load_date") == F.lit(run_date).cast("date"))
print("bronze rows for", run_date, ":", bronze.count())

## 2. Clean and standardise

In [ ]:
cleaned = normalise_nulls(bronze)
cleaned = trim_columns(cleaned, ["customer_name", "city", "street", "state", "region", "district"])
cleaned = rename_columns(cleaned, {"valid_from": "source_valid_from", "valid_to": "source_valid_to"})
cleaned = cast_columns(
    cleaned,
    {
        "customer_id": "bigint",
        "source_valid_from": "bigint",
        "source_valid_to": "bigint",
        "units_purchased": "int",
        "loyalty_segment": "int",
        "lon": "double",
        "lat": "double",
        "tax_id": "string",
        "postcode": "string",
        "number": "string",
        "district": "string",
    },
)
cleaned = strip_float_suffix(cleaned, ["postcode", "number", "district"])

## 3. Derive business columns
`split_part(name, ',', 1)` is the surname because the CRM writes `LAST, FIRST`;
a name without a comma is an organisation.

In [ ]:
enriched = add_derived(
    cleaned,
    {
        "customer_type": "CASE WHEN customer_name LIKE '%,%' THEN 'individual' ELSE 'organisation' END",
        "last_name": "CASE WHEN customer_name LIKE '%,%' THEN trim(split_part(customer_name, ',', 1)) END",
        "first_name": "CASE WHEN customer_name LIKE '%,%' THEN trim(split_part(customer_name, ',', 2)) END",
        "postcode": "nullif(postcode, '0')",
        "source_valid_from_ts": "CAST(source_valid_from AS TIMESTAMP)",
        "source_valid_to_ts": "CAST(source_valid_to AS TIMESTAMP)",
        "is_active": "source_valid_to IS NULL",
        "loyalty_segment_name": "CASE loyalty_segment WHEN 0 THEN 'Bronze' WHEN 1 THEN 'Silver' WHEN 2 THEN 'Gold' WHEN 3 THEN 'Platinum' ELSE 'Unknown' END",
    },
)
enriched = drop_columns(enriched, ["_ingested_at", "_source_file"])

## 4. One row per customer, then merge with history
The hash covers business columns only, so a reload with identical content
creates no new version.

In [ ]:
target = qualified(catalog, silver_schema, "customers")

with track(spark, catalog, run_id, run_date, task="customers_silver", layer="silver", entity="customers") as stats:
    hashed = row_hash(enriched, business_columns(enriched, exclude=["source_valid_from", "source_valid_to"]))
    latest = deduplicate(hashed, keys=["customer_id"], order_by="source_valid_from")
    prepared = latest.withColumn("_updated_at", F.current_timestamp())

    stats.rows_read = prepared.count()
    scd2_merge(spark, prepared, target, keys=["customer_id"], detect_deletes=detect_deletes)
    stats.rows_written = stats.rows_read

    set_table_properties(spark, catalog, silver_schema, "customers")
    cluster_by(spark, catalog, silver_schema, "customers", ["customer_id"])
    log_info(logger, "silver customers merged", rows=stats.rows_read, detect_deletes=detect_deletes)

In [ ]:
display(
    spark.sql(
        f"""
        SELECT is_current, count(*) AS rows, count(DISTINCT customer_id) AS customers
        FROM {catalog}.{silver_schema}.customers GROUP BY is_current
        """
    )
)